In [1]:
!pip install ortools torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.

In [2]:
import os
import re
import pandas as pd
import numpy as np
from ortools.linear_solver import pywraplp
import numpy as np
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
# We use PyTorch Geometric (PyG) for Graph Neural Networks
import itertools
from sklearn.cluster import KMeans
try:
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv
except ImportError:
    print("PyTorch Geometric is not installed. Please run: !pip install torch_geometric")

In [3]:
def load_all_datasets(folder_path):
    """
    Scans the given folder path and automatically loads all distance (C) 
    and demand (wij) matrices into dictionaries based on their node count.
    """
    distance_matrices = {}
    demand_matrices = {}
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None, None

    print(f"Scanning directory: {folder_path}...\n")
    
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        if not os.path.isfile(file_path):
            continue
            
        match = re.search(r'\d+', filename)
        if not match:
            continue
        nodes = int(match.group())
        
        try:
            # 1. Load Distance Matrices (C_ij)
            if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
                df = pd.read_csv(file_path, header=None)
                distance_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes. Shape: {df.values.shape}")
                
            # 2. Load Demand Matrices (W_ij)
            elif filename.startswith('wij'):
                if filename.endswith('.xlsx'):
                    df = pd.read_excel(file_path)
                elif filename.endswith('.csv'):
                    df = pd.read_csv(file_path)
                else:
                    continue
                
                # Ensure the matrix is exactly n x n by dropping extra index column if present
                if df.shape[1] > nodes:
                    df = df.iloc[:, 1:]
                    
                demand_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes. Shape: {df.values.shape}")
                
        except Exception as e:
            print(f"[X] Failed to load {filename}: {e}")

    return distance_matrices, demand_matrices


def calculate_node_production_attraction(W_matrix):
    """
    Calculates the production (O_i) and attraction (D_i) for each node.
    """
    production = np.sum(W_matrix, axis=1) 
    attraction = np.sum(W_matrix, axis=0) 
    return production, attraction


def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
    """
    Generates stochastic demand scenarios (W_ijs).
    Uses a Normal Distribution where mean = nominal demand, 
    and std_dev = variance_factor * nominal demand.
    
    Returns: A 3D NumPy array of shape (n, n, s)
    """
    np.random.seed(seed) # Ensures reproducibility of scenarios
    n = W_matrix.shape[0]
    
    # Generate scenarios
    # Shape of random.normal output will be (num_scenarios, n, n)
    W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
    
    # Demand cannot be negative, so we clip any negative values to 0
    W_scenarios = np.maximum(W_scenarios, 0)
    
    # Transpose to format (i, j, s) as per standard math notation W_ijs
    W_scenarios = np.transpose(W_scenarios, (1, 2, 0))
    
    return W_scenarios


def calculate_path_costs(C_matrix, alpha=0.5):
    """
    Calculates the path cost parameter C_ijkm = C_ik + alpha * C_km + C_mj.
    Uses NumPy broadcasting for extreme efficiency without slow loops.
    
    Returns: A 4D NumPy array of shape (n, n, n, n) where axes are (i, j, k, m)
    """
    n = C_matrix.shape[0]
    
    # C_ik shape: (n, 1, n, 1) -> corresponds to indices (i, j, k, m)
    C_ik = C_matrix[:, np.newaxis, :, np.newaxis]
    
    # C_km shape: (1, 1, n, n) -> corresponds to indices (i, j, k, m)
    C_km = C_matrix[np.newaxis, np.newaxis, :, :]
    
    # C_mj shape: (1, n, 1, n) -> corresponds to indices (i, j, k, m)
    # We transpose C to match (j, m) alignment before reshaping
    C_mj = C_matrix.T[np.newaxis, :, np.newaxis, :]
    
    # Broadcast addition
    C_ijkm = C_ik + (alpha * C_km) + C_mj
    
    return C_ijkm

In [4]:
def solve_robust_hub_location_ortools(C_ijkm, W_ijs, p, gamma=0.05):
    """
    Solves the Risk-Averse (Robust) Single Allocation Hub Location Problem
    using Google OR-Tools (SCIP Solver).
    """
    n = C_ijkm.shape[0]     
    num_s = W_ijs.shape[2]  
    
    # 1. Initialize OR-Tools Solver
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        return None, None
        
    # 2. Decision Variables
    y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(n)}
    z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for i in range(n) for k in range(n)}
    X = {(i, j, k, m): solver.NumVar(0, solver.infinity(), f'X_{i}_{j}_{k}_{m}') 
         for i in range(n) for j in range(n) for k in range(n) for m in range(n)}
                    
    mu = solver.NumVar(0, solver.infinity(), 'mu')
    lam = {s: solver.NumVar(0, solver.infinity(), f'lam_{s}') for s in range(num_s)}
        
    # 3. Objective Function
    risk_coef = 1.0 / (num_s * gamma)
    objective = solver.Objective()
    objective.SetMinimization()
    objective.SetCoefficient(mu, 1.0)
    for s in range(num_s):
        objective.SetCoefficient(lam[s], risk_coef)
        
    # 4. Constraints
    solver.Add(sum(y[k] for k in range(n)) == p, name="select_p_hubs")
    
    for i in range(n):
        solver.Add(sum(z[i, k] for k in range(n)) == 1, name=f"alloc_{i}")
        for k in range(n):
            solver.Add(z[i, k] <= y[k], name=f"valid_alloc_{i}_{k}")
            
    for i in range(n):
        for j in range(n):
            for k in range(n):
                solver.Add(sum(X[i, j, k, m] for m in range(n)) == z[i, k], name=f"f1_{i}_{j}_{k}")
            for m in range(n):
                solver.Add(sum(X[i, j, k, m] for k in range(n)) == z[j, m], name=f"f2_{i}_{j}_{m}")
                
    # Memory-optimized Scenario Violation Constraints
    for s in range(num_s):
        # We construct the expression directly to save RAM instead of building massive dictionaries
        scenario_expr = sum(
            # OR-Tools will implicitly handle the float32 to float64 conversion here
            float(W_ijs[i, j, s]) * float(C_ijkm[i, j, k, m]) * X[i, j, k, m]
            for i in range(n) for j in range(n) for k in range(n) for m in range(n)
        )
        solver.Add(scenario_expr - mu <= lam[s])
        
    # 5. Optimize
    solver.set_time_limit(600000) # 10 minutes max per sample
    status = solver.Solve()
    
    # 6. Extract Results
    if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
        optimal_hubs = [k for k in range(n) if y[k].solution_value() > 0.5]
        return optimal_hubs, solver.Objective().Value()
    else:
        return None, None

def generate_training_labels(C_matrix, W_matrix, num_samples=3, subset_size=10, p=3, num_scenarios=20):
    """
    Randomly samples sub-networks to generate labels (ground truth) for the Machine Learning model.
    This prevents Kaggle from running out of memory.
    """
    print(f"\n--- Starting Label Generation Pipeline ---")
    print(f"Target: Generate {num_samples} samples of size {subset_size} nodes.")
    
    training_data = []
    original_n = C_matrix.shape[0]
    
    for sample_idx in range(num_samples):
        print(f"\nGenerating Sample {sample_idx + 1}/{num_samples}...")
        
        # 1. Randomly select 'subset_size' nodes from the large graph
        node_indices = sorted(random.sample(range(original_n), subset_size))
        
        # 2. Slice the matrices for these specific nodes
        sub_C = C_matrix[np.ix_(node_indices, node_indices)]
        sub_W = W_matrix[np.ix_(node_indices, node_indices)]
        
        # 3. Generate scenarios and costs, casting to float32 to save RAM without overflowing!
        sub_W_ijs = generate_demand_scenarios(sub_W, num_scenarios=num_scenarios).astype(np.float32)
        sub_C_ijkm = calculate_path_costs(sub_C, alpha=0.5).astype(np.float32)
        
        # 4. Solve exactly to find the labels
        start = time.time()
        hubs, cost = solve_robust_hub_location_ortools(sub_C_ijkm, sub_W_ijs, p=p, gamma=0.05)
        
        if hubs:
            # Map the local hub index back to the original graph index
            actual_hubs = [node_indices[h] for h in hubs]
            print(f"[✓] Solved in {time.time() - start:.1f}s | Selected Hubs (Original Indices): {actual_hubs} | CVaR: {cost:.2f}")
            
            # Save the result for Machine Learning
            training_data.append({
                'nodes': node_indices,
                'hubs': actual_hubs
            })
        else:
            print("[X] Failed to find optimal solution within time limit.")
            
    return training_data

In [5]:
def extract_graph_features(C_matrix, W_matrix, node_indices, hubs=None):
    """
    Converts raw distance and demand matrices into PyTorch Graph Data.
    Extracts node features: Production (O_i), Attraction (D_i), and Mean Distance.
    """
    num_nodes = len(node_indices)
    
    # 1. Node Features (X)
    # Production: sum of demands leaving the node
    production = np.sum(W_matrix, axis=1, keepdims=True)
    # Attraction: sum of demands arriving at the node
    attraction = np.sum(W_matrix, axis=0, keepdims=True).T
    # Mean Distance: average distance to all other nodes
    mean_dist = np.mean(C_matrix, axis=1, keepdims=True)
    
    # Normalize features to help the Neural Network learn faster
    production = production / (np.max(production) + 1e-6)
    attraction = attraction / (np.max(attraction) + 1e-6)
    mean_dist = mean_dist / (np.max(mean_dist) + 1e-6)
    
    # Combine into a single feature matrix [num_nodes, 3]
    node_features = np.hstack([production, attraction, mean_dist])
    x = torch.tensor(node_features, dtype=torch.float)
    
    # 2. Edge Index (Connectivity)
    # For Hub Location, the graph is fully connected
    edge_index = []
    edge_weight = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                edge_index.append([i, j])
                # We use the inverse distance as the edge weight (closer nodes have stronger connection)
                weight = 1.0 / (C_matrix[i, j] + 1e-6)
                edge_weight.append(weight)
                
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(edge_weight, dtype=torch.float)
    
    # 3. Target Labels (Y)
    # 1 if the node is a hub, 0 otherwise
    y = torch.zeros((num_nodes, 1), dtype=torch.float)
    if hubs is not None:
        for i, global_idx in enumerate(node_indices):
            if global_idx in hubs:
                y[i] = 1.0
                
    # Create the PyTorch Geometric Data object
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)
    return data

class DLHrNet(nn.Module):
    """
    Deep Learning Hub-Ranker (DLHr) Architecture using Graph Convolutions.
    """
    def __init__(self, num_features=3, hidden_dim=16):
        super(DLHrNet, self).__init__()
        # First Graph Convolutional Layer
        self.conv1 = GCNConv(num_features, hidden_dim)
        # Second Graph Convolutional Layer
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        # Fully connected layer to output a single probability score per node
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_attr
        
        # Pass through first Graph Conv Layer + ReLU activation
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        
        # Pass through second Graph Conv Layer + ReLU
        x = self.conv2(x, edge_index, edge_weight)
        x = F.relu(x)
        
        # Pass through linear layer and apply Sigmoid to get probability (0 to 1)
        out = self.linear(x)
        return torch.sigmoid(out)

def train_dlhr_model(training_labels, C_data, W_data, epochs=100, lr=0.01):
    """
    Trains the Graph Neural Network using the labels generated in Phase 2.
    """
    print("\n--- Starting Phase 3: Training the DLHr Model ---")
    
    # 1. Convert mathematical outputs into PyTorch Graphs
    dataset = []
    for sample in training_labels:
        nodes = sample['nodes']
        hubs = sample['hubs']
        
        # Get the sliced matrices for this sample
        sub_C = C_data[np.ix_(nodes, nodes)]
        sub_W = W_data[np.ix_(nodes, nodes)]
        
        graph_data = extract_graph_features(sub_C, sub_W, nodes, hubs)
        dataset.append(graph_data)
        
    print(f"Prepared {len(dataset)} graphs for training.")
    
    # 2. Initialize Model, Loss Function, and Optimizer
    model = DLHrNet(num_features=3, hidden_dim=16)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss() # Binary Cross Entropy for Probability Output
    
    model.train()
    
    # 3. Training Loop
    for epoch in range(epochs):
        total_loss = 0
        for data in dataset:
            optimizer.zero_grad()
            
            # Predict hub probabilities
            out = model(data)
            
            # Calculate error compared to ground truth (Phase 2 labels)
            loss = criterion(out, data.y)
            
            # Backpropagation
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataset):.4f}")
            
    print("\nTraining Complete! The AI has learned to rank hubs.")
    return model


In [6]:
def get_dl_rankings(model, C_matrix, W_matrix):
    """
    Uses the trained DLHr model to rank all nodes in the network.
    Returns a sorted list of (node_index, probability_score).
    """
    model.eval()
    num_nodes = C_matrix.shape[0]
    all_nodes = list(range(num_nodes))
    
    # Extract graph features for the full dataset (no hubs provided since we are predicting)
    graph_data = extract_graph_features(C_matrix, W_matrix, all_nodes)
    
    # Get predictions from the Neural Network
    import torch
    with torch.no_grad():
        probabilities = model(graph_data).numpy().flatten()
        
    # Create a ranked list of tuples: (node_id, score) sorted highest to lowest
    ranked_nodes = [(i, float(probabilities[i])) for i in range(num_nodes)]
    ranked_nodes.sort(key=lambda x: x[1], reverse=True)
    
    return ranked_nodes, probabilities

def evaluate_robust_cost(hubs, C_ijkm, W_ijs, gamma=0.05):
    """
    Calculates the Robust (CVaR) cost for a specific set of selected hubs.
    This acts as the 'Fitness Function' for the heuristic.
    """
    n = C_ijkm.shape[0]
    num_s = W_ijs.shape[2]
    
    # 1. Allocation: Assign each node to its nearest open hub
    allocation = np.zeros(n, dtype=int)
    for i in range(n):
        # Find the hub k that minimizes the distance C_ik
        distances_to_hubs = [C_ijkm[i, i, k, k] for k in hubs] # simplified distance
        best_hub_idx = np.argmin(distances_to_hubs)
        allocation[i] = hubs[best_hub_idx]
        
    # 2. Calculate Scenario Costs
    scenario_costs = np.zeros(num_s)
    for s in range(num_s):
        total_cost = 0
        for i in range(n):
            for j in range(n):
                k = allocation[i]
                m = allocation[j]
                # W_ijs * C_ijkm
                total_cost += W_ijs[i, j, s] * C_ijkm[i, j, k, m]
        scenario_costs[s] = total_cost
        
    # 3. Calculate CVaR (Conditional Value at Risk)
    # CVaR is the average cost of the worst (gamma * 100)% scenarios
    scenario_costs.sort()
    worst_cases = scenario_costs[int((1 - gamma) * num_s):]
    
    if len(worst_cases) == 0:
        return np.max(scenario_costs)
        
    return np.mean(worst_cases)

def dl_cbs_heuristic(C_matrix, W_matrix, C_ijkm, W_ijs, probabilities, p=3, candidates_per_cluster=2):
    """
    Phase 9: Deep Learning Clustering-Based Search (DL-CBS).
    Reduces the search space by clustering nodes and picking only high-ranking nodes.
    """
    print(f"\n--- Starting DL-CBS Heuristic (Searching for {p} Hubs) ---")
    start_time = time.time()
    n = C_matrix.shape[0]
    
    # Step 1: Cluster nodes based on geographical distance (C_matrix)
    # We create 'p' clusters so we can pick candidates from different regions
    kmeans = KMeans(n_clusters=p, random_state=42, n_init=10)
    labels = kmeans.fit_predict(C_matrix)
    
    # Step 2: Pick top candidates from each cluster based on DL rankings
    candidate_pools = []
    for cluster_id in range(p):
        nodes_in_cluster = np.where(labels == cluster_id)[0]
        
        # Sort nodes in this cluster by their DL probability score
        cluster_ranked = sorted(nodes_in_cluster, key=lambda x: probabilities[x], reverse=True)
        
        # Keep only the top 'candidates_per_cluster'
        top_candidates = cluster_ranked[:candidates_per_cluster]
        candidate_pools.append(top_candidates)
        print(f"Cluster {cluster_id} Top Candidates: {top_candidates}")
        
    # Step 3: Generate combinations taking exactly one candidate from each cluster
    combinations = list(itertools.product(*candidate_pools))
    print(f"Reduced search space to {len(combinations)} combinations to evaluate.")
    
    # Step 4: Evaluate the combinations to find the robust minimum
    best_hubs = None
    best_cost = float('inf')
    
    for combo in combinations:
        cost = evaluate_robust_cost(list(combo), C_ijkm, W_ijs, gamma=0.05)
        if cost < best_cost:
            best_cost = cost
            best_hubs = list(combo)
            
    print(f"\n[DL-CBS Completed in {time.time() - start_time:.2f}s]")
    print(f"Best Hubs Found: {best_hubs}")
    print(f"Robust Cost (CVaR): {best_cost:.2f}")
    
    return best_hubs, best_cost

In [7]:
def run_full_pipeline_test():
    """
    Executes the entire Machine Learning + Heuristic pipeline sequentially 
    to ensure all variables are passed correctly.
    """
    print("="*50)
    print("🚀 STARTING FULL PIPELINE TEST")
    print("="*50)
    
    # ---------------------------------------------------------
    # Phase 1: Data Loading
    # ---------------------------------------------------------
    dataset_folder = r"/kaggle/input/datasets/infernalss/node-dataset"
    C_data, W_data = load_all_datasets(dataset_folder)
    
    if 25 not in C_data or 25 not in W_data:
        print("[Error] 25-node dataset not found. Please check paths.")
        return
        
    C_25 = C_data[25]
    W_25 = W_data[25]
    
    # ---------------------------------------------------------
    # Phase 2: Exact Solver Label Generation (Ground Truth)
    # ---------------------------------------------------------
    print("\n[PHASE 2] Generating Training Data with Exact Solver...")
    # We use 3 samples of 10 nodes for a quick test. 
    # (Increase num_samples to 20 or 50 for better AI accuracy later!)
    training_labels = generate_training_labels(
        C_matrix=C_25, 
        W_matrix=W_25, 
        num_samples=3, 
        subset_size=10, 
        p=3, 
        num_scenarios=20
    )
    
    if not training_labels:
        print("[Error] Failed to generate training labels. Stopping.")
        return
        
    # ---------------------------------------------------------
    # Phase 3: Train the DLHr Graph Neural Network
    # ---------------------------------------------------------
    print("\n[PHASE 3] Training Deep Learning Hub-Ranker (DLHr)...")
    # This trains the model and explicitly returns it so we can use it below
    trained_model = train_dlhr_model(
        training_labels=training_labels, 
        C_data=C_25, 
        W_data=W_25, 
        epochs=100, 
        lr=0.01
    )
    
    # ---------------------------------------------------------
    # Phase 9: DL-Augmented Heuristic (DL-CBS)
    # ---------------------------------------------------------
    print("\n[PHASE 9] Executing DL-CBS on Full 25-Node Network...")
    
    # Step 9.1: Use the trained model to rank all 25 nodes
    ranked_nodes, probabilities = get_dl_rankings(trained_model, C_25, W_25)
    print(f"🌟 Top 5 AI Predicted Hubs: {[n for n, score in ranked_nodes[:5]]}")
    
    # Step 9.2: Prepare full datasets for evaluating combinations
    W_ijs_full = generate_demand_scenarios(W_25, num_scenarios=20).astype(np.float32)
    C_ijkm_full = calculate_path_costs(C_25, alpha=0.5).astype(np.float32)
    
    # Step 9.3: Run DL-CBS
    best_hubs, best_cost = dl_cbs_heuristic(
        C_matrix=C_25, 
        W_matrix=W_25, 
        C_ijkm=C_ijkm_full, 
        W_ijs=W_ijs_full, 
        probabilities=probabilities, 
        p=3, 
        candidates_per_cluster=2
    )
    
    print("\n" + "="*50)
    print("✅ PIPELINE TEST COMPLETE")
    print("="*50)

# Execute the main function
if __name__ == "__main__":
    run_full_pipeline_test()

🚀 STARTING FULL PIPELINE TEST
Scanning directory: /kaggle/input/datasets/infernalss/node-dataset...

[✓] Loaded Demand Matrix (W) for 100 nodes. Shape: (100, 100)
[✓] Loaded Distance Matrix (C) for 200 nodes. Shape: (200, 200)
[✓] Loaded Demand Matrix (W) for 200 nodes. Shape: (200, 200)
[✓] Loaded Demand Matrix (W) for 150 nodes. Shape: (150, 150)
[✓] Loaded Distance Matrix (C) for 150 nodes. Shape: (150, 150)
[✓] Loaded Distance Matrix (C) for 100 nodes. Shape: (100, 100)
[✓] Loaded Demand Matrix (W) for 25 nodes. Shape: (25, 25)
[✓] Loaded Distance Matrix (C) for 25 nodes. Shape: (25, 25)

[PHASE 2] Generating Training Data with Exact Solver...

--- Starting Label Generation Pipeline ---
Target: Generate 3 samples of size 10 nodes.

Generating Sample 1/3...
[✓] Solved in 9.0s | Selected Hubs (Original Indices): [3, 6, 11] | CVaR: 775031547.56

Generating Sample 2/3...
[✓] Solved in 67.7s | Selected Hubs (Original Indices): [7, 11, 24] | CVaR: 1164852706.23

Generating Sample 3/3...
